In [26]:
import pandas as pd
import numpy as np

In [ ]:
travel_times_file = "/Users/sierrawelsch/CS4100_Final/Datasets/2022-Q1_HRTravelTimes.csv"
events_file = "/Users/sierrawelsch/CS4100_Final/Datasets/2022-01_HREvents.csv"
# expected arr/dep times for 2022-01-04
stop_times_file = "/Users/sierrawelsch/CS4100_Final/Datasets/20220103_stop_times.txt"

In [34]:
events_df = pd.read_csv(events_file)
events_df.head()
events_df = events_df.drop(["vehicle_id", "vehicle_label", "direction_id"], axis = 1)
events_df = events_df[events_df['service_date'] == '2022-01-04']
events_df['event_time_dt'] = pd.to_timedelta(events_df['event_time_sec'], unit='s').astype(str)
events_df

,service_date,route_id,trip_id,stop_id,stop_sequence,event_type,event_time,event_time_sec,event_time_dt
78707,2022-01-04,Red,50034550,70094,50,ARR,1641291477,19077,0 days 05:17:57
78708,2022-01-04,Red,50034550,70094,50,DEP,1641292105,19705,0 days 05:28:25
78709,2022-01-04,Red,50034550,70092,60,ARR,1641292251,19851,0 days 05:30:51
78710,2022-01-04,Red,50034550,70092,60,DEP,1641292293,19893,0 days 05:31:33
78711,2022-01-04,Red,50034550,70090,70,ARR,1641292359,19959,0 days 05:32:39
...,...,...,...,...,...,...,...,...,...
111238,2022-01-04,Red,NONREV-1580420617,70066,200,ARR,1641355406,83006,0 days 23:03:26
111239,2022-01-04,Red,NONREV-1580420617,70066,200,DEP,1641355453,83053,0 days 23:04:13
111240,2022-01-04,Red,NONREV-1580420617,70064,210,ARR,1641355529,83129,0 days 23:05:29
111241,2022-01-04,Red,NONREV-1580420617,70068,190,ARR,1641355225,82825,0 days 23:00:25


In [9]:
travel_times_df = pd.read_csv(travel_times_file)
travel_times_df.head()
travel_times_df = travel_times_df.drop(["direction_id"], axis = 1)
travel_times_df

,service_date,from_stop_id,to_stop_id,route_id,start_time_sec,end_time_sec,travel_time_sec
0,2022-01-01,70004,70001,Orange,45822,46027,205
1,2022-01-01,70004,70001,Orange,45355,45556,201
2,2022-01-01,70004,70001,Orange,58824,59108,284
3,2022-01-01,70004,70001,Orange,57792,58021,229
4,2022-01-01,70004,70001,Orange,44557,44776,219
...,...,...,...,...,...,...,...
10441190,2022-03-31,70041,70838,Blue,28550,28706,156
10441191,2022-03-31,70041,70838,Blue,27646,27776,130
10441192,2022-03-31,70041,70838,Blue,28326,28469,143
10441193,2022-03-31,70041,70838,Blue,27181,27316,135


In [35]:
stop_times_df = pd.read_csv(stop_times_file)
stop_times_df = stop_times_df.drop(["stop_headsign", "continuous_pickup", "continuous_drop_off"], axis=1)
stop_times_df = stop_times_df.dropna()
stop_times_df['departure_time_sec'] = stop_times_df['departure_time'].apply(lambda x: sum(int(i) * 60**(2 - idx) for idx, i in enumerate(x.split(':'))))
stop_times_df['arrival_time_sec'] = stop_times_df['arrival_time'].apply(lambda x: sum(int(i) * 60**(2 - idx) for idx, i in enumerate(x.split(':'))))

stop_times_df

/var/folders/z7/152736054bv0sl_wm9ygspfr0000gn/T/ipykernel_35930/962061595.py:1: DtypeWarning: Columns (0: trip_id, 1: stop_id, 2: stop_headsign) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times_df = pd.read_csv(stop_times_file)


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,pickup_type,drop_off_type,timepoint,checkpoint_id,departure_time_sec,arrival_time_sec
0,49812229,24:30:00,24:30:00,70036,1,0,1,1,ogmnl,88200,88200
1,49812229,24:32:00,24:32:00,70034,10,0,0,1,mlmnl,88320,88320
2,49812229,24:35:00,24:35:00,70032,20,0,0,1,welln,88500,88500
3,49812229,24:37:00,24:37:00,70278,30,0,0,1,astao,88620,88620
4,49812229,24:40:00,24:40:00,70030,40,0,0,1,sull,88800,88800
...,...,...,...,...,...,...,...,...,...,...,...
1818621,50533146,16:20:00,16:20:00,3582,17,1,3,0,hngdpt,58800,58800
1818622,50533147,11:50:00,11:50:00,111146,1,3,1,1,ppth,42600,42600
1818638,50533147,12:20:00,12:20:00,3582,17,1,3,0,hngdpt,44400,44400
1818639,50533148,09:50:00,09:50:00,111146,1,3,1,1,ppth,35400,35400


In [36]:
merged_df = pd.merge(
    events_df, 
    stop_times_df, 
    on=['trip_id', 'stop_id', 'stop_sequence'],
    how='left'
)
merged_df = merged_df.dropna()
merged_df = merged_df.drop(['drop_off_type', 'pickup_type', 'timepoint'], axis=1)
merged_df

,service_date,route_id,trip_id,stop_id,stop_sequence,event_type,event_time,event_time_sec,event_time_dt,arrival_time,departure_time,checkpoint_id,departure_time_sec,arrival_time_sec
0,2022-01-04,Red,50034550,70094,50,ARR,1641291477,19077,0 days 05:17:57,05:16:00,05:16:00,asmnl,18960.0,18960.0
1,2022-01-04,Red,50034550,70094,50,DEP,1641292105,19705,0 days 05:28:25,05:16:00,05:16:00,asmnl,18960.0,18960.0
2,2022-01-04,Red,50034550,70092,60,ARR,1641292251,19851,0 days 05:30:51,05:18:00,05:18:00,smmnl,19080.0,19080.0
3,2022-01-04,Red,50034550,70092,60,DEP,1641292293,19893,0 days 05:31:33,05:18:00,05:18:00,smmnl,19080.0,19080.0
4,2022-01-04,Red,50034550,70090,70,ARR,1641292359,19959,0 days 05:32:39,05:20:00,05:20:00,fldcr,19200.0,19200.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19884,2022-01-04,Orange,50040727,70005,20,DEP,1641296063,23663,0 days 06:34:23,06:32:00,06:32:00,sbmnl,23520.0,23520.0
19885,2022-01-04,Orange,50040727,70003,10,DEP,1641295952,23552,0 days 06:32:32,06:30:00,06:30:00,grnst,23400.0,23400.0
19886,2022-01-04,Orange,50040727,70003,10,ARR,1641295928,23528,0 days 06:32:08,06:30:00,06:30:00,grnst,23400.0,23400.0
19887,2022-01-04,Orange,50040727,70001,1,DEP,1641295842,23442,0 days 06:30:42,06:29:00,06:29:00,forhl,23340.0,23340.0


In [37]:
merged_df['delay_sec'] = np.where(
    merged_df['event_type'] == 'ARR',
    merged_df['event_time_sec'] - merged_df['arrival_time_sec'],
    merged_df['event_time_sec'] - merged_df['departure_time_sec']
)

# Optionally convert to minutes
merged_df['delay_min'] = merged_df['delay_sec'] / 60
merged_df

,service_date,route_id,trip_id,stop_id,stop_sequence,event_type,event_time,event_time_sec,event_time_dt,arrival_time,departure_time,checkpoint_id,departure_time_sec,arrival_time_sec,delay_sec,delay_min
0,2022-01-04,Red,50034550,70094,50,ARR,1641291477,19077,0 days 05:17:57,05:16:00,05:16:00,asmnl,18960.0,18960.0,117.0,1.950000
1,2022-01-04,Red,50034550,70094,50,DEP,1641292105,19705,0 days 05:28:25,05:16:00,05:16:00,asmnl,18960.0,18960.0,745.0,12.416667
2,2022-01-04,Red,50034550,70092,60,ARR,1641292251,19851,0 days 05:30:51,05:18:00,05:18:00,smmnl,19080.0,19080.0,771.0,12.850000
3,2022-01-04,Red,50034550,70092,60,DEP,1641292293,19893,0 days 05:31:33,05:18:00,05:18:00,smmnl,19080.0,19080.0,813.0,13.550000
4,2022-01-04,Red,50034550,70090,70,ARR,1641292359,19959,0 days 05:32:39,05:20:00,05:20:00,fldcr,19200.0,19200.0,759.0,12.650000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19884,2022-01-04,Orange,50040727,70005,20,DEP,1641296063,23663,0 days 06:34:23,06:32:00,06:32:00,sbmnl,23520.0,23520.0,143.0,2.383333
19885,2022-01-04,Orange,50040727,70003,10,DEP,1641295952,23552,0 days 06:32:32,06:30:00,06:30:00,grnst,23400.0,23400.0,152.0,2.533333
19886,2022-01-04,Orange,50040727,70003,10,ARR,1641295928,23528,0 days 06:32:08,06:30:00,06:30:00,grnst,23400.0,23400.0,128.0,2.133333
19887,2022-01-04,Orange,50040727,70001,1,DEP,1641295842,23442,0 days 06:30:42,06:29:00,06:29:00,forhl,23340.0,23340.0,102.0,1.700000
